In [ ]:
# v49 — KTF3: P1 Mirror Correlation — eBOSS DR16 Quasar Catalog
#
# Quasars are the deepest available tracer before Euclid.
# eBOSS DR16 QSO: ~500,000 quasars, z=0.8-2.2
# Comoving depth: up to ~6000 Mpc -- well beyond T1 = 1660 Mpc
#
# KTF3 Prediction P1: mirror cross-correlation peak at r = T1/2 = 830 Mpc
# Mirror operation: phi(r) = r - 2*(r.n_hat)*n_hat
# KTF3 axis: l=277, b=-31 (galactic coordinates)
#
# PRE-REGISTERED: March 2026
# Cotting, S. (2026) — academia.edu/AndrewCotting
# GitHub: github.com/Andrew-Cot/KTF3-Analyse
# AI assistance declared (Claude, Anthropic). All scientific ideas: author's own.

In [ ]:
# CELL 1 -- Imports and KTF3 axis
!pip install numpy matplotlib scipy astropy -q
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
import os, warnings
warnings.filterwarnings('ignore')

cosmo = FlatLambdaCDM(H0=67.4, Om0=0.315)

# KTF3 preferred axis
L_KTF3, B_KTF3 = 277.0, -31.0
l_rad = np.radians(L_KTF3)
b_rad = np.radians(B_KTF3)
n_hat = np.array([
    np.cos(b_rad) * np.cos(l_rad),
    np.cos(b_rad) * np.sin(l_rad),
    np.sin(b_rad)
])

print('v49 -- KTF3 P1 Mirror Correlation: eBOSS DR16 Quasars')
print('PRE-REGISTERED March 2026 -- Cotting, S.')
print('='*60)
print()
print('Catalog: eBOSS DR16 QSO (~500k quasars, z=0.8-2.2)')
print('Depth: up to ~6000 Mpc -- deepest available before Euclid')
print('KTF3 axis: l =', L_KTF3, 'b =', B_KTF3)
print('Prediction: peak at r = T1/2 = 830 Mpc')

In [ ]:
# CELL 2 -- Download eBOSS DR16 QSO catalog
QSO_FILE = 'eboss_dr16_qso.fits'
QSO_URL  = 'https://data.sdss.org/sas/dr16/eboss/lss/catalogs/DR16/eBOSS_QSO_clustering_data-NGC-vDR16.fits'

if not os.path.exists(QSO_FILE):
    print('Downloading eBOSS DR16 QSO catalog...')
    os.system('wget -q -O ' + QSO_FILE + ' "' + QSO_URL + '"')
    sz = os.path.getsize(QSO_FILE) if os.path.exists(QSO_FILE) else 0
    print('Size MB:', round(sz/1e6, 1))
    if sz < 1000000:
        print('NGC file too small, trying SGC...')
        QSO_URL_SGC = 'https://data.sdss.org/sas/dr16/eboss/lss/catalogs/DR16/eBOSS_QSO_clustering_data-SGC-vDR16.fits'
        os.system('wget -q -O ' + QSO_FILE + ' "' + QSO_URL_SGC + '"')
        sz = os.path.getsize(QSO_FILE) if os.path.exists(QSO_FILE) else 0
        print('Size MB:', round(sz/1e6, 1))
else:
    print('File present:', QSO_FILE)
    print('Size MB:', round(os.path.getsize(QSO_FILE)/1e6, 1))

# Load
hdul = fits.open(QSO_FILE)
data = hdul[1].data
print('Columns:', data.dtype.names)
print('Objects:', len(data))

In [ ]:
# CELL 3 -- Prepare catalog in galactic coordinates
ra_eq  = data['RA'].astype(float)
dec_eq = data['DEC'].astype(float)
z      = data['Z'].astype(float)

# Convert to galactic
coords = SkyCoord(ra=ra_eq*u.deg, dec=dec_eq*u.deg, frame='icrs')
gal    = coords.galactic
l_deg  = gal.l.deg
b_deg  = gal.b.deg

# Quality cuts
good = (z > 0.5) & (z < 2.5) & np.isfinite(z)
l_c, b_c, z_c = l_deg[good], b_deg[good], z[good]

print('After cuts:', len(l_c), 'quasars')
print('z range:', round(z_c.min(),3), 'to', round(z_c.max(),3))
print('Median z:', round(float(np.median(z_c)),3))

# Convert to comoving Cartesian in galactic frame
chi    = cosmo.comoving_distance(z_c).value
l_rad_arr = np.radians(l_c)
b_rad_arr = np.radians(b_c)
x = chi * np.cos(b_rad_arr) * np.cos(l_rad_arr)
y = chi * np.cos(b_rad_arr) * np.sin(l_rad_arr)
z_cart = chi * np.sin(b_rad_arr)
pos = np.stack([x, y, z_cart], axis=1)

# KTF3-correct mirror
proj = pos @ n_hat
pos_mirror = pos - 2 * np.outer(proj, n_hat)

print('Comoving range:', round(chi.min()), 'to', round(chi.max()), 'Mpc')
print('T1/2 = 830 Mpc well within range')

In [ ]:
# CELL 4 -- Mirror cross-correlation
T1     = 1660
r_pred = T1 / 2  # 830 Mpc

bins = np.arange(200, 2001, 100)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
N_BINS = len(bin_centers)

N_PAIRS = 500000
N_gal = len(pos)
rng = np.random.default_rng(42)
idx1 = rng.integers(0, N_gal, N_PAIRS)
idx2 = rng.integers(0, N_gal, N_PAIRS)
keep = idx1 != idx2
idx1, idx2 = idx1[keep], idx2[keep]

r_std    = np.linalg.norm(pos[idx2] - pos[idx1], axis=1)
r_mirror = np.linalg.norm(pos[idx2] - pos_mirror[idx1], axis=1)

N_total  = len(idx1)
N_std    = np.zeros(N_BINS)
N_mirror = np.zeros(N_BINS)
for b in range(N_BINS):
    N_std[b]    = ((r_std    >= bins[b]) & (r_std    < bins[b+1])).sum()
    N_mirror[b] = ((r_mirror >= bins[b]) & (r_mirror < bins[b+1])).sum()

C_std    = N_std    / N_total
C_mirror = N_mirror / N_total
C_excess = C_mirror - C_std

peak_r = bin_centers[np.argmax(C_excess)]
print('Mirror excess peak at: r =', peak_r, 'Mpc')
print('Predicted peak at:     r =', r_pred, 'Mpc')

In [ ]:
# CELL 5 -- Monte Carlo significance
N_MC = 300
print('Monte Carlo null (N=' + str(N_MC) + ')...')

mc_excess = np.zeros((N_MC, N_BINS))
rng_mc = np.random.default_rng(99)

for i in range(N_MC):
    theta = rng_mc.uniform(0, np.pi)
    phi   = rng_mc.uniform(0, 2*np.pi)
    n_rand = np.array([np.sin(theta)*np.cos(phi),
                       np.sin(theta)*np.sin(phi),
                       np.cos(theta)])
    proj_r = pos @ n_rand
    pos_rand_mirror = pos - 2 * np.outer(proj_r, n_rand)
    r_m = np.linalg.norm(pos_rand_mirror[idx1] - pos[idx2], axis=1)
    N_m = np.zeros(N_BINS)
    for b in range(N_BINS):
        N_m[b] = ((r_m >= bins[b]) & (r_m < bins[b+1])).sum()
    mc_excess[i] = N_m/N_total - C_std
    if (i+1) % 75 == 0:
        print('  MC', i+1, '/', N_MC)

mc_mean = mc_excess.mean(axis=0)
mc_std  = mc_excess.std(axis=0)
sigma   = (C_excess - mc_mean) / (mc_std + 1e-10)

pred_bin   = np.argmin(np.abs(bin_centers - r_pred))
sigma_pred = sigma[pred_bin]
print('Sigma at r =', r_pred, 'Mpc: sigma =', round(sigma_pred, 2))
print('Max sigma:', round(sigma.max(), 2), 'at r =', bin_centers[sigma.argmax()], 'Mpc')

In [ ]:
# CELL 6 -- Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

ax = axes[0]
ax.fill_between(bin_centers, mc_mean-2*mc_std, mc_mean+2*mc_std,
                alpha=0.3, color='gray', label='2sigma null')
ax.plot(bin_centers, C_excess, color='#AB47BC', lw=2, label='Mirror excess')
ax.axvline(r_pred, color='yellow', lw=2, ls='--', label='T1/2=830 Mpc')
ax.axhline(0, color='white', lw=0.5, alpha=0.4)
ax.set_xlabel('r [Mpc]', color='white')
ax.set_ylabel('C_mirror - C_std', color='white')
ax.set_title('P1: Mirror Correlation (KTF3-axis)\neBOSS DR16 Quasars', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

ax = axes[1]
ax.plot(bin_centers, sigma, color='#AB47BC', lw=2)
ax.axhline(0,  color='white',  lw=0.5, alpha=0.4)
ax.axhline(2,  color='orange', lw=1.5, ls='--', alpha=0.7, label='2sigma')
ax.axhline(-2, color='orange', lw=1.5, ls='--', alpha=0.7)
ax.axvline(r_pred, color='yellow', lw=2, ls='--', label='Predicted: 830 Mpc')
ax.set_xlabel('r [Mpc]', color='white')
ax.set_ylabel('Significance sigma', color='white')
ax.set_title('P1 Significance\nsigma at 830 Mpc = ' + str(round(sigma_pred,2)), color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

plt.suptitle('v49: KTF3 P1 -- eBOSS DR16 QSO -- Cotting (2026)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('v49_eboss_qso_mirror.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# CELL 7 -- Summary
print('\n' + '='*60)
print('RESULT v49: KTF3 P1 MIRROR CORRELATION -- eBOSS DR16 QSO')
print('Cotting, S. (2026) -- PRE-REGISTERED March 2026')
print('='*60)
print('  Catalog: eBOSS DR16 Quasars (spectroscopic)')
print('  Objects:', N_gal)
print('  Mirror axis: l =', L_KTF3, 'b =', B_KTF3)
print()
print('  Predicted peak: r = 830 Mpc')
print('  Observed peak:  r =', peak_r, 'Mpc')
print('  Sigma at r=830: ', round(sigma_pred, 2))
print()
if sigma_pred > 2:
    verdict = 'CONFIRMED -- Peak at predicted location.'
elif sigma_pred > 1:
    verdict = 'MARGINAL -- Weak excess.'
elif sigma_pred > 0:
    verdict = 'WEAK -- Correct sign, not significant.'
else:
    verdict = 'NULL -- No mirror excess at predicted location.'
print('  Verdict:', verdict)
print()
print('  P1 Mirror Correlation -- Complete Summary:')
print('  BOSS DR12 (v48b):  sigma = +0.13  (25% sky, z<0.7)')
print('  eBOSS QSO (v49):   sigma =', round(sigma_pred,2), ' (z=0.8-2.2)')
print('  Euclid DR1 (v44b): sigma = TBD    (October 2026)')
print()
print('  Pre-registration: academia.edu/AndrewCotting')
print('  GitHub: github.com/Andrew-Cot/KTF3-Analyse')
print('='*60)